# Data Preprocessing — Panel Dataset Construction

This notebook downloads raw indicators from Eurostat, cleans them and reshapes each one into a country × sector × year panel. The result is a set of per-variable CSV files plus two master panels used by the subsequent regression models.

**Pipeline overview**

1. Setup & helper functions
2. AI adoption
3. ICT training
4. ICT specialists
5. Wages (nominal → real)
6. Productivity (GVA per employee)
7. Firm size
8. Education
9. High skills
10. Master panel — core variables
11. Macroeconomic controls (unemployment, inflation, GDP per capita)
12. Full dataset


## 1. Setup

Imports, pandas display settings, the country list (EU + EFTA) and a helper that maps NACE codes to one-letter section codes.


In [110]:
import pandas as pd
import numpy as np
import re
import eurostat 

In [ ]:
pd.set_option("display.max_rows", None)      
pd.set_option("display.max_columns", None)   
pd.set_option("display.width", None)         
pd.set_option("display.max_colwidth", None)  

In [112]:
EU_COUNTRIES = [
    'BE', 'BG', 'CZ', 'DK', 'DE', 'EE', 'IE', 
    'EL', 'ES', 'FR', 'HR', 'IT', 'CY', 'LV',
    'LT', 'LU', 'HU', 'MT', 'NL', 'AT', 'PL', 
    'PT', 'RO', 'SI', 'SK', 'FI', 'SE'
]

EFTA_COUNTRIES = ['IS', 'LI', 'NO', 'CH']

EU_EFTA = EU_COUNTRIES + EFTA_COUNTRIES

In [113]:
def nace_section_or_nan(s: str) -> str | float:
    s = str(s).strip().upper()
    match = re.fullmatch(r'([A-U])', s)

    if match:
        return match.group(1)
    else:
        return np.nan

## 2. AI Adoption

Share of enterprises using at least one AI technology (`isoc_eb_ain2`). The survey is available for 2021, 2023 and 2025, so the missing **2022** observation is filled in as the average of 2021 and 2023.


In [115]:
ain2 = eurostat.get_data_df('isoc_eb_ain2')
ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
ain2.describe()
ain2.describe(include=["object", "bool"])

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\3664687001.py:2: SyntaxWarning: invalid escape sequence '\T'
  ain2.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


,freq,size_emp,nace_r2,indic_is,unit,geo
count,236998,236998,236998,236998,236998,236998
unique,1,1,50,63,6,36
top,A,GE10,C-E,E_AI_BINC,PC_ENT,PL
freq,236998,236998,4844,5359,95165,8138


In [116]:
# Filter the data 
# E_AI_TANY - Enterprises use at least one of the AI technologies

ai_adopt = ain2.copy()

ai_adopt = ai_adopt[
    (ai_adopt["indic_is"] == "E_AI_TANY") &
    (ai_adopt["unit"] == "PC_ENT") & 
    (ai_adopt["size_emp"] == "GE10")
]

ai_adopt = ai_adopt.drop(columns=['indic_is','unit',  'freq', 'size_emp'])

ai_adopt = ai_adopt[ai_adopt['geo'].isin(EU_EFTA)]
print(ai_adopt)

              nace_r2 geo   2021   2023    2024    2025
3848                C  AT   9.61  12.31   22.71   32.54
3850                C  BE  10.42  15.31   23.24   39.80
3851                C  BG   2.88   2.55    4.35    5.25
3852                C  CY   2.05   3.81    2.98    4.26
3853                C  CZ   4.19   6.01    9.55   16.73
3854                C  DE   9.19   9.32   16.11   24.38
3855                C  DK  27.28  14.80   21.75   38.55
3857                C  EE   2.72   3.12    9.77   22.66
3858                C  EL   5.35   5.19    7.48    7.77
3859                C  ES   7.42   8.68    9.81   17.14
3861                C  FI  16.25  12.98   20.75     NaN
3862                C  FR   5.33   4.87    7.43   16.93
3863                C  HR   8.29   6.70    9.71   12.90
3864                C  HU   4.00   4.24    4.83    7.78
3865                C  IE   8.83   7.18   17.51   17.66
3866                C  IT   6.63   4.94    8.03   14.73
3867                C  LT   5.06   4.83    8.81 

In [117]:
check_S = ai_adopt[ai_adopt["nace_r2"] == "S"].count()
print(check_S)

nace_r2    0
geo        0
2021       0
2023       0
2024       0
2025       0
dtype: int64


In [118]:
ai_adopt['nace_r2_1d'] = ai_adopt['nace_r2'].map(nace_section_or_nan)
print(ai_adopt)

              nace_r2 geo   2021   2023    2024    2025 nace_r2_1d
3848                C  AT   9.61  12.31   22.71   32.54          C
3850                C  BE  10.42  15.31   23.24   39.80          C
3851                C  BG   2.88   2.55    4.35    5.25          C
3852                C  CY   2.05   3.81    2.98    4.26          C
3853                C  CZ   4.19   6.01    9.55   16.73          C
3854                C  DE   9.19   9.32   16.11   24.38          C
3855                C  DK  27.28  14.80   21.75   38.55          C
3857                C  EE   2.72   3.12    9.77   22.66          C
3858                C  EL   5.35   5.19    7.48    7.77          C
3859                C  ES   7.42   8.68    9.81   17.14          C
3861                C  FI  16.25  12.98   20.75     NaN          C
3862                C  FR   5.33   4.87    7.43   16.93          C
3863                C  HR   8.29   6.70    9.71   12.90          C
3864                C  HU   4.00   4.24    4.83    7.78       

In [11]:
ai_adopt = ai_adopt.dropna()
print(pd.unique(ai_adopt['nace_r2']))
print(ai_adopt)

['C' 'D' 'E' 'F' 'G' 'H' 'I' 'J' 'L' 'M' 'N']
       nace_r2 geo   2021   2023   2024   2025 nace_r2_1d
3848         C  AT   9.61  12.31  22.71  32.54          C
3850         C  BE  10.42  15.31  23.24  39.80          C
3851         C  BG   2.88   2.55   4.35   5.25          C
3852         C  CY   2.05   3.81   2.98   4.26          C
3853         C  CZ   4.19   6.01   9.55  16.73          C
3854         C  DE   9.19   9.32  16.11  24.38          C
3855         C  DK  27.28  14.80  21.75  38.55          C
3857         C  EE   2.72   3.12   9.77  22.66          C
3858         C  EL   5.35   5.19   7.48   7.77          C
3859         C  ES   7.42   8.68   9.81  17.14          C
3862         C  FR   5.33   4.87   7.43  16.93          C
3863         C  HR   8.29   6.70   9.71  12.90          C
3864         C  HU   4.00   4.24   4.83   7.78          C
3865         C  IE   8.83   7.18  17.51  17.66          C
3866         C  IT   6.63   4.94   8.03  14.73          C
3867         C  LT   5.06 

In [12]:
# Create the 2022 column by averaging 2021 and 2023
ai_adopt['2022'] = (ai_adopt['2021'] + ai_adopt['2023']) / 2

# Check the results
print(ai_adopt.head())

     nace_r2 geo   2021   2023   2024   2025 nace_r2_1d    2022
3848       C  AT   9.61  12.31  22.71  32.54          C  10.960
3850       C  BE  10.42  15.31  23.24  39.80          C  12.865
3851       C  BG   2.88   2.55   4.35   5.25          C   2.715
3852       C  CY   2.05   3.81   2.98   4.26          C   2.930
3853       C  CZ   4.19   6.01   9.55  16.73          C   5.100


In [13]:
ai_adopt = ai_adopt.drop(columns='nace_r2_1d')
cols_to_melt = ['2021', '2022', '2023', '2024', '2025']

df_ai_panel = ai_adopt.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,        
    var_name='year_raw',             
    value_name='ai_adoption'        
)


df_ai_panel['year'] = df_ai_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ai_panel.drop(columns=['year_raw'], inplace=True)
df_ai_panel = df_ai_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ai_panel.head(10))

  geo nace_r2  ai_adoption  year
0  AT       C         9.61  2021
1  AT       C        10.96  2022
2  AT       C        12.31  2023
3  AT       C        22.71  2024
4  AT       C        32.54  2025
5  AT       F         3.12  2021
6  AT       F         3.70  2022
7  AT       F         4.28  2023
8  AT       F         7.35  2024
9  AT       F        14.89  2025


In [14]:
df_ai_panel.to_csv('data_panel/ai_adopt.csv', index = False)

## 3. ICT Training

Enterprises that provided ICT training to their personnel (`isoc_ske_ittn2`). Available for 2020, 2022 and 2024, so it enters the model as a **lagged** regressor.


In [ ]:
train = eurostat.get_data_df('isoc_ske_ittn2')
print(train)


In [16]:
train.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
train.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16918 entries, 0 to 16917
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      16918 non-null  object 
 1   size_emp  16918 non-null  object 
 2   nace_r2   16918 non-null  object 
 3   indic_is  16918 non-null  object 
 4   unit      16918 non-null  object 
 5   geo       16918 non-null  object 
 6   2020      5499 non-null   float64
 7   2022      7278 non-null   float64
 8   2024      7386 non-null   float64
dtypes: float64(3), object(6)
memory usage: 1.2+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\2930950165.py:1: SyntaxWarning: invalid escape sequence '\T'
  train.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [17]:
# Filter the data 
train_new = train.copy()
train_new = train_new[
    (train_new['indic_is'] == 'E_ITT2') & # Enterprise provided training to their personnel to develop their ICT skills
    (train_new['unit'] == 'PC_ENT') & # Percentage of enterprises 
    (train_new['size_emp'] == 'GE10')
]

train_new = train_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
train_new = train_new[train_new['geo'].isin(EU_EFTA)]

In [18]:
train_new['nace_r2_1d'] = train_new['nace_r2'].map(nace_section_or_nan)
train_new.drop(columns=['nace_r2'], inplace=True) 
train_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
train_new = train_new.dropna(subset=['nace_r2'])
print(train_new)

      geo   2020   2022    2024 nace_r2
239    AT  20.37  25.60   26.06       C
241    BE    NaN  33.89   40.12       C
242    BG   5.38   6.47    5.82       C
243    CY  19.69  20.51   17.41       C
244    CZ  27.60  23.76   28.35       C
245    DE  26.58  27.82   28.73       C
246    DK  28.78  30.45   34.28       C
248    EE  14.63  13.27   15.78       C
249    EL    NaN  14.46   13.70       C
250    ES  18.04  16.99   19.15       C
255    FI  38.88  40.57   35.48       C
256    FR  15.91  16.57   14.60       C
257    HR  19.05  18.26   16.92       C
258    HU  17.25  17.55   17.44       C
259    IE  31.80  28.40   29.00       C
260    IS    NaN    NaN     NaN       C
261    IT  15.53  18.97   20.01       C
262    LT  12.73  11.20   13.77       C
263    LU  18.81  29.85   28.34       C
264    LV  12.59  11.71    9.71       C
267    MT  19.62  21.64   28.55       C
268    NL  25.35  28.63   27.81       C
269    NO  33.46  36.60   31.32       C
270    PL  16.14  22.75   29.21       C


In [19]:
from sklearn.impute import KNNImputer

year_cols = ['2020', '2022', '2024']
missing_masks = {col: train_new[col].isna() for col in year_cols}


def impute_by_sector(group):
    n_rows = len(group)
    if n_rows > 1:
        neighbors = min(5, n_rows - 1)
        sector_imputer = KNNImputer(n_neighbors=neighbors, weights='distance')
        
        valid_cols = group[year_cols].dropna(axis=1, how='all').columns
        if len(valid_cols) > 0:
            group[valid_cols] = sector_imputer.fit_transform(group[valid_cols])
            
    return group

train_new = train_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)

# impute the 2020 year using macroeconomic deflation ratios for remaining missing values
country_means = train_new.groupby('geo')[['2020', '2022']].mean()

# Calculate the deflation ratio (Country Avg 2020 / Country Avg 2022)
country_means['deflation_ratio'] = np.where(
    country_means['2022'] != 0, 
    country_means['2020'] / country_means['2022'], 
    np.nan
)

ratio_map = country_means['deflation_ratio'].to_dict()
train_new['macro_ratio'] = train_new['geo'].map(ratio_map)

# Apply the formula ONLY where 2020 is STILL missing and 2022 exists
mask = train_new['2020'].isna() & train_new['2022'].notna()
train_new.loc[mask, '2020'] = train_new.loc[mask, '2022'] * train_new.loc[mask, 'macro_ratio']
train_new = train_new.drop(columns=['macro_ratio'])

for col in year_cols:
    train_new[f'is_{col}_imputed'] = (missing_masks[col] & train_new[col].notna()).astype(int)

C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\2462806180.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  train_new = train_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)


In [20]:
print(train_new)

      geo       2020       2022        2024 nace_r2  is_2020_imputed  \
239    AT  20.370000  25.600000   26.060000       C                0   
241    BE  34.672128  33.890000   40.120000       C                1   
242    BG   5.380000   6.470000    5.820000       C                0   
243    CY  19.690000  20.510000   17.410000       C                0   
244    CZ  27.600000  23.760000   28.350000       C                0   
245    DE  26.580000  27.820000   28.730000       C                0   
246    DK  28.780000  30.450000   34.280000       C                0   
248    EE  14.630000  13.270000   15.780000       C                0   
249    EL  14.791065  14.460000   13.700000       C                1   
250    ES  18.040000  16.990000   19.150000       C                0   
255    FI  38.880000  40.570000   35.480000       C                0   
256    FR  15.910000  16.570000   14.600000       C                0   
257    HR  19.050000  18.260000   16.920000       C             

In [21]:
cols_to_melt = ['2020', '2022', '2024']

df_train_panel = train_new.melt(
    id_vars=['geo', 'nace_r2'],      
    value_vars=cols_to_melt,         
    var_name='year_raw',            
    value_name='training_ict'        
)

df_train_panel['year'] = df_train_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_train_panel.drop(columns=['year_raw'], inplace=True)
df_train_panel['training_ict'] = df_train_panel['training_ict'].round(2)
df_train_panel = df_train_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print(df_train_panel.head(10))

  geo nace_r2  training_ict  year
0  AT       C         20.37  2020
1  AT       C         25.60  2022
2  AT       C         26.06  2024
3  AT       D         41.66  2020
4  AT       D         50.06  2022
5  AT       D         48.11  2024
6  AT       E         23.76  2020
7  AT       E         28.55  2022
8  AT       E         26.70  2024
9  AT       F          9.65  2020


In [22]:
df_train_panel.to_csv('data_panel/train.csv', index = False)

## 4. ICT Specialists

Enterprises employing ICT specialists (`isoc_ske_itspen2`). Available for 2020, 2022 and 2024, so it enters the model as a **lagged** regressor.


In [ ]:
ict_spec = eurostat.get_data_df('isoc_ske_itspen2')
ict_spec.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
print(ict_spec)


In [24]:
ict_spec.drop(['2012', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
ict_spec.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3417 entries, 0 to 3416
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      3417 non-null   object 
 1   size_emp  3417 non-null   object 
 2   nace_r2   3417 non-null   object 
 3   indic_is  3417 non-null   object 
 4   unit      3417 non-null   object 
 5   geo       3417 non-null   object 
 6   2020      1137 non-null   float64
 7   2022      1480 non-null   float64
 8   2024      1497 non-null   float64
dtypes: float64(3), object(6)
memory usage: 240.4+ KB


In [25]:
# Filter the data 
# E_ITSP2 - Enterprise employed ICT/IT specialists (reduced comparability with 2007)

ict_spec_new = ict_spec.copy()
ict_spec_new = ict_spec_new[
    (ict_spec_new['unit'] == 'PC_ENT') & # Percentage of enterprises 
    (ict_spec_new['size_emp'] == 'GE10')
]

ict_spec_new = ict_spec_new.drop(columns=['freq', 'size_emp', 'indic_is', 'unit'])
ict_spec_new = ict_spec_new[ict_spec_new['geo'].isin(EU_EFTA)]

ict_spec_new['nace_r2_1d'] = ict_spec_new['nace_r2'].map(nace_section_or_nan)

ict_spec_new.drop(columns=['nace_r2'], inplace=True) 
ict_spec_new.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
ict_spec_new = ict_spec_new.dropna(subset=['nace_r2'])
print(ict_spec_new)

     geo   2020   2022    2024 nace_r2
1     AT  28.74  28.37   30.02       C
3     BE    NaN  35.46   35.31       C
4     BG  15.16  14.73   15.53       C
5     CY  14.23  16.29   14.93       C
6     CZ  21.66  21.42   23.05       C
7     DE  24.97  25.82   26.81       C
8     DK  30.07  31.99   26.96       C
10    EE  14.83  14.37   16.71       C
11    EL    NaN  17.08   19.06       C
12    ES  17.03  13.05   13.90       C
17    FI  34.04  33.07   33.73       C
18    FR  19.15  20.45   17.55       C
19    HR  16.61  15.15   14.29       C
20    HU  30.72  34.97   30.38       C
21    IE  33.48  36.91   38.11       C
22    IS  12.10    NaN     NaN       C
23    IT  12.96  14.02   13.64       C
24    LT  15.30  17.45   16.15       C
25    LU  24.05  25.83   24.56       C
26    LV  17.61  17.52   16.83       C
29    MT  24.54  28.48   29.29       C
30    NL  25.52  31.64   30.69       C
31    NO  18.80  19.85   17.77       C
32    PL  27.28  32.05   25.87       C
33    PT  18.18  20.11   

In [26]:
year_cols = ['2020', '2022', '2024']
missing_masks = {col: ict_spec_new[col].isna() for col in year_cols}

def impute_by_sector(group):
    n_rows = len(group)
    if n_rows > 1:
        neighbors = min(5, n_rows - 1)
        sector_imputer = KNNImputer(n_neighbors=neighbors, weights='distance')
        
        valid_cols = group[year_cols].dropna(axis=1, how='all').columns
        if len(valid_cols) > 0:
            group[valid_cols] = sector_imputer.fit_transform(group[valid_cols])
            
    return group

ict_spec_new = ict_spec_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)

# impute the 2020 year using macroeconomic deflation ratios for remaining missing values
country_means = ict_spec_new.groupby('geo')[['2020', '2022']].mean()

# Calculate the deflation ratio (Country Avg 2020 / Country Avg 2022)
country_means['deflation_ratio'] = np.where(
    country_means['2022'] != 0, 
    country_means['2020'] / country_means['2022'], 
    np.nan
)

ratio_map = country_means['deflation_ratio'].to_dict()
ict_spec_new['macro_ratio'] = ict_spec_new['geo'].map(ratio_map)

# Apply the formula ONLY where 2020 is STILL missing and 2022 exists
mask = ict_spec_new['2020'].isna() & ict_spec_new['2022'].notna()
ict_spec_new.loc[mask, '2020'] = ict_spec_new.loc[mask, '2022'] * ict_spec_new.loc[mask, 'macro_ratio']
ict_spec_new = ict_spec_new.drop(columns=['macro_ratio'])

for col in year_cols:
    ict_spec_new[f'is_{col}_imputed'] = (missing_masks[col] & ict_spec_new[col].notna()).astype(int)

C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\4217061009.py:16: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ict_spec_new = ict_spec_new.groupby('nace_r2', group_keys=False).apply(impute_by_sector)


In [27]:
print(ict_spec_new)

     geo       2020       2022        2024 nace_r2  is_2020_imputed  \
1     AT  28.740000  28.370000   30.020000       C                0   
3     BE  31.541380  35.460000   35.310000       C                1   
4     BG  15.160000  14.730000   15.530000       C                0   
5     CY  14.230000  16.290000   14.930000       C                0   
6     CZ  21.660000  21.420000   23.050000       C                0   
7     DE  24.970000  25.820000   26.810000       C                0   
8     DK  30.070000  31.990000   26.960000       C                0   
10    EE  14.830000  14.370000   16.710000       C                0   
11    EL  17.710641  17.080000   19.060000       C                1   
12    ES  17.030000  13.050000   13.900000       C                0   
17    FI  34.040000  33.070000   33.730000       C                0   
18    FR  19.150000  20.450000   17.550000       C                0   
19    HR  16.610000  15.150000   14.290000       C                0   
20    

In [28]:
cols_to_melt = ['2020','2022', '2024']

df_ict_panel = ict_spec_new.melt(
    id_vars=['geo', 'nace_r2'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='spec_ict'           
)

df_ict_panel['year'] = df_ict_panel['year_raw'].str.extract(r'(\d+)').astype(int)
df_ict_panel.drop(columns=['year_raw'], inplace=True)
df_ict_panel['spec_ict'] = df_ict_panel['spec_ict'].round(2)
df_ict_panel = df_ict_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)
print(df_ict_panel.head(10))

  geo nace_r2  spec_ict  year
0  AT       C     28.74  2020
1  AT       C     28.37  2022
2  AT       C     30.02  2024
3  AT       D     40.63  2020
4  AT       D     48.06  2022
5  AT       D     47.21  2024
6  AT       E     23.40  2020
7  AT       E     27.69  2022
8  AT       E     25.20  2024
9  AT       F      8.24  2020


In [29]:
df_ict_panel.to_csv('data_panel/ict_spec.csv', index = False)

## 5. Wages


### 5.1 Nominal wages (EUR per hour)


In [30]:
lc = eurostat.get_data_df('lc_lci_lev')
lc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
lc.info()

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\4158338232.py:2: SyntaxWarning: invalid escape sequence '\T'
  lc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12213 entries, 0 to 12212
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      12213 non-null  object 
 1   unit      12213 non-null  object 
 2   lcstruct  12213 non-null  object 
 3   nace_r2   12213 non-null  object 
 4   geo       12213 non-null  object 
 5   2008      6054 non-null   float64
 6   2012      7672 non-null   float64
 7   2016      8222 non-null   float64
 8   2020      10475 non-null  float64
 9   2021      10763 non-null  float64
 10  2022      10767 non-null  float64
 11  2023      10771 non-null  float64
 12  2024      10701 non-null  float64
 13  2025      9846 non-null   float64
dtypes: float64(9), object(5)
memory usage: 1.3+ MB


In [31]:
lc.drop(['2008', '2012','2016'], axis='columns', inplace=True)

# Filter the data 
# per employee in full-time equivalents, per hour

wg = lc.copy()
wg = wg[
    (wg['unit'] == 'EUR') &
    (wg['lcstruct'] == 'D11') # Wages and Salaries (total)
]

wg = wg.drop(columns=['freq', 'unit', 'lcstruct'])
wg = wg[wg['geo'].isin(EU_EFTA)]

print(wg)

     nace_r2 geo  2020  2021  2022  2023  2024  2025
1          B  AT  28.4  28.8  29.2  31.7  34.7  36.4
3          B  BE  31.3  31.7  34.0  36.7  37.6   NaN
4          B  BG   8.1   8.8  10.6  11.4  12.2  13.4
5          B  CH   NaN   NaN   NaN   NaN   NaN   NaN
6          B  CY  13.6  14.9  15.0  16.4  17.1  17.5
7          B  CZ  11.2  11.8  13.3  14.9  14.8  15.8
8          B  DE  31.7  32.2  35.1  37.5  39.4  38.9
9          B  DK  57.9   NaN   NaN   NaN   NaN   NaN
13         B  EE  11.3  12.1  13.1  14.3  15.2  16.0
14         B  EL  11.5  11.3  10.7  10.8  13.0  13.8
15         B  ES  25.1  25.4  23.4  23.8  24.6  25.5
19         B  FI  32.7  32.0  32.5  34.0  34.9  35.5
20         B  FR  26.5  26.9  27.6  28.6  29.4  30.0
21         B  HR  10.8  10.4  10.4  11.9  13.1  14.8
22         B  HU   8.7   8.8   9.7  11.3  13.1  14.2
23         B  IE  27.9  29.0  33.2  34.9  34.8  35.5
24         B  IS   NaN   NaN   NaN   NaN   NaN   NaN
25         B  IT  32.0  31.3  32.2  32.5  31.5

In [32]:
wg['nace_r2_1d'] = wg['nace_r2'].map(nace_section_or_nan)

wg.drop(columns=['nace_r2'], inplace=True) 
wg.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

wg = wg.rename(columns={'2020': 'wg_n_2020',
                        '2021': 'wg_n_2021',
                        '2022' : 'wg_n_2022',
                        '2023' : 'wg_n_2023',
                        '2024' : 'wg_n_2024',
                        '2025' : 'wg_n_2025'})
print(wg)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024  wg_n_2025  \
1    AT       28.4       28.8       29.2       31.7       34.7       36.4   
3    BE       31.3       31.7       34.0       36.7       37.6        NaN   
4    BG        8.1        8.8       10.6       11.4       12.2       13.4   
5    CH        NaN        NaN        NaN        NaN        NaN        NaN   
6    CY       13.6       14.9       15.0       16.4       17.1       17.5   
7    CZ       11.2       11.8       13.3       14.9       14.8       15.8   
8    DE       31.7       32.2       35.1       37.5       39.4       38.9   
9    DK       57.9        NaN        NaN        NaN        NaN        NaN   
13   EE       11.3       12.1       13.1       14.3       15.2       16.0   
14   EL       11.5       11.3       10.7       10.8       13.0       13.8   
15   ES       25.1       25.4       23.4       23.8       24.6       25.5   
19   FI       32.7       32.0       32.5       34.0       34.9       35.5   

### 5.2 HICP (2015 = 100)


In [ ]:
hicp = eurostat.get_data_df('prc_hicp_aind')
print(hicp)

In [34]:
hicp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1996, 2020)]
hicp.drop(columns=years_to_drop, errors='ignore', inplace=True)
hicp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35287 entries, 0 to 35286
Data columns (total 10 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    35287 non-null  object 
 1   unit    35287 non-null  object 
 2   coicop  35287 non-null  object 
 3   geo     35287 non-null  object 
 4   2020    32509 non-null  float64
 5   2021    32624 non-null  float64
 6   2022    32564 non-null  float64
 7   2023    32540 non-null  float64
 8   2024    32542 non-null  float64
 9   2025    32535 non-null  float64
dtypes: float64(6), object(4)
memory usage: 2.7+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\908363276.py:1: SyntaxWarning: invalid escape sequence '\T'
  hicp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [35]:
# Filter the data 
# annual average index

hicp_inx = hicp.copy()
hicp_inx = hicp_inx.reset_index(drop=True)
hicp_inx = hicp_inx[
    (hicp_inx['unit'] == 'INX_A_AVG') &
    (hicp_inx['coicop'] == 'CP00') 
]

hicp_inx = hicp_inx.drop(columns=['freq', 'unit', 'coicop'])
hicp_inx = hicp_inx[hicp_inx['geo'].isin(EU_EFTA)]

print(hicp_inx)

    geo    2020    2021    2022    2023    2024    2025
233  AT  108.47  111.46  121.07  130.40  134.21  139.01
234  BE  108.23  111.71  123.26  126.07  131.52  135.49
235  BG  106.27  109.30  123.52  134.15  137.63  142.50
236  CH  100.56  101.04  103.74  106.10  107.25  107.36
237  CY   99.67  101.92  110.17  114.50  117.09  118.06
238  CZ  111.40  115.10  132.10  147.90  151.90  155.40
239  DE  105.80  109.20  118.70  125.90  129.00  131.90
240  DK  102.90  104.90  113.80  117.60  119.10  121.30
244  EE  109.80  114.72  137.03  149.52  155.10  162.57
246  EL  101.17  101.75  111.21  115.84  119.31  122.75
247  ES  103.91  107.04  115.95  119.89  123.33  126.65
251  FI  103.98  106.12  113.74  118.67  119.83  122.01
252  FR  105.50  107.68  114.04  120.50  123.29  124.43
253  HR  103.06  105.82  117.11  126.94  132.04  137.82
254  HU  113.15  119.04  137.22  160.59  166.56  173.96
255  IE  101.20  103.60  112.00  117.80  119.40  121.90
256  IS  103.06  106.84  112.96  121.96  127.47 

In [36]:
cols_to_melt = ['2020', '2021', '2022', '2023', '2024', '2025']

hicp_panel = hicp_inx.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='hicp'           
)

hicp_panel['year'] = hicp_panel['year_raw'].str.extract(r'(\d+)').astype(int)
hicp_panel.drop(columns=['year_raw'], inplace=True)
hicp_panel = hicp_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(hicp_panel.head(10))

  geo    hicp  year
0  AT  108.47  2020
1  AT  111.46  2021
2  AT  121.07  2022
3  AT  130.40  2023
4  AT  134.21  2024
5  AT  139.01  2025
6  BE  108.23  2020
7  BE  111.71  2021
8  BE  123.26  2022
9  BE  126.07  2023


In [37]:
hicp_panel.to_csv('data_panel/hicp.csv', index = False)

In [38]:
hicp_inx = hicp_inx.rename(columns={'2020': 'hicp_inx_2020',
                        '2021': 'hicp_inx_2021',
                        '2022' : 'hicp_inx_2022',
                        '2023' : 'hicp_inx_2023',
                        '2024' : 'hicp_inx_2024', 
                        '2025' : 'hicp_inx_2025'})

### 5.3 Real wages (2021–2025)


In [39]:
wage_countries = set(wg['geo'].unique())
hicp_countries = set(hicp_inx['geo'].unique())

missing_countries = wage_countries - hicp_countries

if len(missing_countries) > 0:
    print(f"WARNING: The following countries are in the Wage data but MISSING in HICP data:\n{missing_countries}")
else:
    print("All countries in Wage data have a matching HICP record.")

All countries in Wage data have a matching HICP record.


In [40]:
wg_merged = pd.merge(wg, hicp_inx, on='geo', how='left')

In [41]:
years = ['2020', '2021', '2022', '2023', '2024', '2025']

for year in years:
    # Define column names
    nom_col = f'wg_n_{year}'  
    hicp_col = f'hicp_inx_{year}'  
    real_col = f'wg_r_{year}'  
    
    # Apply formula
    wg_merged[real_col] = (wg_merged[nom_col] / wg_merged[hicp_col]) * 100

print(wg_merged)

    geo  wg_n_2020  wg_n_2021  wg_n_2022  wg_n_2023  wg_n_2024  wg_n_2025  \
0    AT       28.4       28.8       29.2       31.7       34.7       36.4   
1    BE       31.3       31.7       34.0       36.7       37.6        NaN   
2    BG        8.1        8.8       10.6       11.4       12.2       13.4   
3    CH        NaN        NaN        NaN        NaN        NaN        NaN   
4    CY       13.6       14.9       15.0       16.4       17.1       17.5   
5    CZ       11.2       11.8       13.3       14.9       14.8       15.8   
6    DE       31.7       32.2       35.1       37.5       39.4       38.9   
7    DK       57.9        NaN        NaN        NaN        NaN        NaN   
8    EE       11.3       12.1       13.1       14.3       15.2       16.0   
9    EL       11.5       11.3       10.7       10.8       13.0       13.8   
10   ES       25.1       25.4       23.4       23.8       24.6       25.5   
11   FI       32.7       32.0       32.5       34.0       34.9       35.5   

In [42]:
wg_merged = wg_merged.drop(['wg_n_2020', 'wg_n_2021', 'wg_n_2022', 'wg_n_2023', 'wg_n_2024', 'wg_n_2025',
                            'hicp_inx_2020', 'hicp_inx_2021', 'hicp_inx_2022', 'hicp_inx_2023', 'hicp_inx_2024', 'hicp_inx_2025'], axis='columns')
wg_merged = wg_merged.dropna()
print(wg_merged)

    geo nace_r2  wg_r_2020  wg_r_2021  wg_r_2022  wg_r_2023  wg_r_2024  \
0    AT       B  26.182355  25.838866  24.118279  24.309816  25.855003   
2    BG       B   7.622095   8.051235   8.581606   8.497950   8.864346   
4    CY       B  13.645029  14.619309  13.615322  14.323144  14.604151   
5    CZ       B  10.053860  10.251955  10.068130  10.074375   9.743252   
6    DE       B  29.962193  29.487179  29.570345  29.785544  30.542636   
8    EE       B  10.291439  10.547420   9.559950   9.563938   9.800129   
9    EL       B  11.367006  11.105651   9.621437   9.323204  10.895985   
10   ES       B  24.155519  23.729447  20.181113  19.851531  19.946485   
11   FI       B  31.448355  30.154542  28.573941  28.650881  29.124593   
12   FR       B  25.118483  24.981426  24.202034  23.734440  23.846216   
13   HR       B  10.479332   9.828010   8.880540   9.374508   9.921236   
14   HU       B   7.688909   7.392473   7.068940   7.036553   7.865034   
15   IE       B  27.569170  27.992278 

In [43]:
wg_panel = wg_merged.melt(
    id_vars=['geo', 'nace_r2'],    
    value_vars=['wg_r_2020', 'wg_r_2021', 'wg_r_2022', 'wg_r_2023', 'wg_r_2024', 'wg_r_2025'], 
    var_name='year_raw',            
    value_name='real_wage'          
)

wg_panel['year'] = wg_panel['year_raw'].str.extract(r'(\d+)').astype(int)
wg_panel.drop(columns=['year_raw'], inplace=True)
wg_panel = wg_panel.sort_values(by=['geo', 'nace_r2', 'year']).reset_index(drop=True)

print("Panel Format Ready:")
print(wg_panel.head(10))

Panel Format Ready:
  geo nace_r2  real_wage  year
0  AT       B  26.182355  2020
1  AT       B  25.838866  2021
2  AT       B  24.118279  2022
3  AT       B  24.309816  2023
4  AT       B  25.855003  2024
5  AT       B  26.185167  2025
6  AT       C  27.104268  2020
7  AT       C  26.736049  2021
8  AT       C  26.183200  2022
9  AT       C  26.073620  2023


In [44]:
wg_panel.to_csv('data_panel/wage.csv', index = False)

## 6. Productivity


### 6.1 Gross value added (GVA)


In [147]:
gva = eurostat.get_data_df('nama_10_a64')

In [148]:
gva.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
gva.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 293304 entries, 0 to 293303
Data columns (total 56 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   freq     293304 non-null  object 
 1   unit     293304 non-null  object 
 2   nace_r2  293304 non-null  object 
 3   na_item  293304 non-null  object 
 4   geo      293304 non-null  object 
 5   1975     15231 non-null   float64
 6   1976     15597 non-null   float64
 7   1977     15598 non-null   float64
 8   1978     20361 non-null   float64
 9   1979     20952 non-null   float64
 10  1980     29448 non-null   float64
 11  1981     30544 non-null   float64
 12  1982     30547 non-null   float64
 13  1983     30547 non-null   float64
 14  1984     30559 non-null   float64
 15  1985     30563 non-null   float64
 16  1986     30567 non-null   float64
 17  1987     30559 non-null   float64
 18  1988     30561 non-null   float64
 19  1989     30561 non-null   float64
 20  1990     30558 non-null   

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\1352719858.py:1: SyntaxWarning: invalid escape sequence '\T'
  gva.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [149]:
years_to_drop = [str(year) for year in range(1975, 2021)]
gva = gva.drop(columns=years_to_drop, errors='ignore')
gva['nace_r2_1d'] = gva['nace_r2'].map(nace_section_or_nan)
gva.drop(columns=['nace_r2'], inplace=True) 
gva.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
gva = gva[gva['geo'].isin(EU_EFTA)]

In [150]:
gva = gva[
    (gva['na_item'] == 'B1G') &         # Gross value added
    (gva['unit'] == 'CP_MEUR')        # Current prices, million EUR
]
# print(gva)

### 6.2 Employees


In [ ]:
emp_df = eurostat.get_data_df('nama_10_a64_e')

emp = emp_df[
    (emp_df['na_item'] == 'EMP_DC') &     # Employment, domestic concept
    (emp_df['unit'] == 'THS_PER')       # Thousands of persons
]
print(emp)

In [152]:
emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
years_to_drop = [str(year) for year in range(1975, 2021)]
emp = emp.drop(columns=years_to_drop, errors='ignore')
emp['nace_r2_1d'] = emp['nace_r2'].map(nace_section_or_nan)
emp.drop(columns=['nace_r2'], inplace=True) 
emp.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
emp = emp[emp['geo'].isin(EU_EFTA)]
print(emp)

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\3286445458.py:1: SyntaxWarning: invalid escape sequence '\T'
  emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\3286445458.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  emp.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


      freq     unit na_item geo      2021      2022      2023      2024  \
33726    A  THS_PER  EMP_DC  AT    155.35    149.98    138.12    136.50   
33728    A  THS_PER  EMP_DC  BE     60.30     59.50     58.20     56.60   
33729    A  THS_PER  EMP_DC  BG    532.61    521.84    517.02    488.91   
33730    A  THS_PER  EMP_DC  CH    121.24    114.99    122.48    125.16   
33731    A  THS_PER  EMP_DC  CY     15.80     15.99     16.09     16.32   
33732    A  THS_PER  EMP_DC  CZ    147.04    154.17    154.12    153.05   
33733    A  THS_PER  EMP_DC  DE    579.00    577.00    570.00    565.00   
33734    A  THS_PER  EMP_DC  DK     65.39     65.75     63.29     62.65   
33739    A  THS_PER  EMP_DC  EE     17.78     17.31     18.75     19.40   
33740    A  THS_PER  EMP_DC  EL    504.60    508.30    518.70    508.40   
33741    A  THS_PER  EMP_DC  ES    804.20    797.90    777.00    769.90   
33743    A  THS_PER  EMP_DC  FI     87.70     87.00     82.90     80.30   
33744    A  THS_PER  EMP_

### 6.3 Labour productivity (GVA per employee)


In [ ]:
gva = gva[['geo', 'nace_r2', '2021', '2022', '2023', '2024', '2025']].copy()
emp = emp[['geo', 'nace_r2', '2021', '2022', '2023', '2024', '2025']].copy()

gva = gva.rename(columns={str(y): f'gva_{y}' for y in [2021, 2022, 2023, 2024, 2025]})
emp = emp.rename(columns={str(y): f'emp_{y}' for y in [2021, 2022, 2023, 2024, 2025]})

prod = gva.merge(emp, on=['geo', 'nace_r2'], how='inner')

for y in [2021, 2022, 2023, 2024, 2025]:
    prod[f'gva_{y}'] = pd.to_numeric(prod[f'gva_{y}'], errors='coerce')
    prod[f'emp_{y}'] = pd.to_numeric(prod[f'emp_{y}'], errors='coerce')

print(prod)

In [154]:
# productivity (EUR per employee) for each year
for y in [2021, 2022, 2023, 2024, 2025]:
    gva_col = f'gva_{y}'    # million EUR
    emp_col = f'emp_{y}'    # thousand persons
    prod_col = f'prod_{y}'  # EUR per employee

    prod[prod_col] = (
        prod[gva_col]     # in thousand EUR
    ) / (
        prod[emp_col]          # persons
    )

    # Clean impossible values
    prod[prod_col] = prod[prod_col].replace([np.inf, -np.inf], np.nan)


print(prod.head())
print("\nCoverage:")
print("Countries:", prod['geo'].nunique())
print("NACE sectors:", prod['nace_r2'].unique())

  geo nace_r2  gva_2021  gva_2022  gva_2023  gva_2024  gva_2025  emp_2021  \
0  AT       A    4936.7    6050.9    5933.6    6011.2       NaN    155.35   
1  BE       A    3314.1    3768.8    4665.8    4910.3       NaN     60.30   
2  BG       A    3088.4    3194.7    2384.3    2463.1       NaN    532.61   
3  CH       A    4363.3    4878.7    5231.0    5463.8       NaN    121.24   
4  CY       A     385.8     351.6     374.7     401.6       NaN     15.80   

   emp_2022  emp_2023  emp_2024  emp_2025  prod_2021  prod_2022  prod_2023  \
0    149.98    138.12    136.50       NaN  31.777921  40.344713  42.959745   
1     59.50     58.20     56.60       NaN  54.960199  63.341176  80.168385   
2    521.84    517.02    488.91       NaN   5.798614   6.121991   4.611620   
3    114.99    122.48    125.16    116.88  35.988948  42.427168  42.709014   
4     15.99     16.09     16.32       NaN  24.417722  21.988743  23.287756   

   prod_2024  prod_2025  
0  44.038095        NaN  
1  86.754417    

In [155]:
prod_long = prod.melt(
    id_vars=['geo', 'nace_r2'],
    value_vars=[f'prod_{y}' for y in [2021, 2022, 2023, 2024, 2025]],
    var_name='year',
    value_name='productivity'
)
prod_long['year'] = prod_long['year'].str.replace('prod_', '').astype(int)
prod_long = prod_long.dropna(subset=['productivity', 'nace_r2'])
prod_long['productivity'] = round(prod_long['productivity'],2)
print(prod_long.head())

  geo nace_r2  year  productivity
0  AT       A  2021         31.78
1  BE       A  2021         54.96
2  BG       A  2021          5.80
3  CH       A  2021         35.99
4  CY       A  2021         24.42


In [161]:
print(prod_long[(prod_long['geo'] == 'IT') & (prod_long['nace_r2'] == 'J')])

       geo nace_r2  year  productivity
89834   IT       J  2021         95.89
258018  IT       J  2022         94.69
426202  IT       J  2023         95.43
594386  IT       J  2024         99.48
762570  IT       J  2025        103.78


In [162]:
prod_long.to_csv('data_panel/product.csv', index = False)

## 7. Firm Size

Share of employment in firms with 250 or more employees (`sbs_sc_ovw`), used as a firm-size proxy.


In [ ]:
fsi =  eurostat.get_data_df('sbs_sc_ovw') 
print(fsi)

In [56]:
fsi.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
fsi['nace_r2_1d'] = fsi['nace_r2'].map(nace_section_or_nan)
fsi.drop(columns=['nace_r2'], inplace=True) 
fsi.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)
fsi = fsi[fsi['geo'].isin(EU_EFTA)]
fsi.info()

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\2874492891.py:1: SyntaxWarning: invalid escape sequence '\T'
  fsi.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
Index: 1218936 entries, 1 to 1410267
Data columns (total 9 columns):
 #   Column     Non-Null Count    Dtype  
---  ------     --------------    -----  
 0   freq       1218936 non-null  object 
 1   indic_sbs  1218936 non-null  object 
 2   size_emp   1218936 non-null  object 
 3   geo        1218936 non-null  object 
 4   2021       899374 non-null   float64
 5   2022       929789 non-null   float64
 6   2023       925820 non-null   float64
 7   2024       301936 non-null   float64
 8   nace_r2    49568 non-null    object 
dtypes: float64(4), object(5)
memory usage: 93.0+ MB


In [57]:
# FSI Calculation (2021-2024) 

df_emp = fsi[fsi['indic_sbs'] == 'EMP_NR'].copy()
df_fsi = df_emp[df_emp['size_emp'].isin(['GE250', 'TOTAL'])].copy()

# Transform year columns into rows
df_melted = df_fsi.melt(
    id_vars=['geo', 'nace_r2', 'size_emp'],     
    value_vars=['2021', '2022', '2023', '2024'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

df_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='size_emp',
    values='emp_value',
    aggfunc='sum'
).reset_index()

df_pivoted.rename(columns={'GE250': 'Emp_250_plus', 'TOTAL': 'Emp_Total'}, inplace=True)

df_pivoted['FSI'] = (df_pivoted['Emp_250_plus'] / df_pivoted['Emp_Total']) * 100

df_fsi_final = df_pivoted[['geo', 'nace_r2', 'year', 'FSI']].copy()

print("FSI Calculation Head (All Years):")
print(df_fsi_final.head())

FSI Calculation Head (All Years):
size_emp geo nace_r2  year        FSI
0         AT       B  2021   0.000000
1         AT       B  2022   0.000000
2         AT       B  2023   0.000000
3         AT       B  2024  39.214390
4         AT       C  2021  55.824158


In [58]:
df_fsi_final.to_csv('data_panel/fsi.csv', index = False)

## 8. Education

Share of the workforce with tertiary education, ISCED levels 5–8 (`edat_lfs_9910`).


In [59]:
educ =  eurostat.get_data_df('edat_lfs_9910') 

In [60]:
educ.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
educ.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
educ.info()

<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\84112933.py:1: SyntaxWarning: invalid escape sequence '\T'
  educ.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 243374 entries, 0 to 243373
Data columns (total 13 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   freq     243374 non-null  object 
 1   unit     243374 non-null  object 
 2   nace_r2  243374 non-null  object 
 3   isced11  243374 non-null  object 
 4   age      243374 non-null  object 
 5   sex      243374 non-null  object 
 6   geo      243374 non-null  object 
 7   2020     114504 non-null  float64
 8   2021     157244 non-null  float64
 9   2022     158344 non-null  float64
 10  2023     156048 non-null  float64
 11  2024     156073 non-null  float64
 12  2025     154031 non-null  float64
dtypes: float64(6), object(7)
memory usage: 24.1+ MB


In [61]:
# Filter the data 
# unit - PC, percent

educ = educ.copy()
educ = educ[
    (educ['sex'] == 'T')& # for all genders
    (educ['age'] == 'Y18-69')& 
    (educ['isced11'] == 'ED5-8')   # 5-8: Tertiary education (levels 5-8)
]

educ = educ.drop(columns=['freq', 'unit', 'age', 'sex', 'isced11'])
educ = educ[educ['geo'].isin(EU_EFTA)]
educ = educ.dropna()

In [62]:
education_long = educ.melt(
    id_vars=['nace_r2', 'geo'],
    value_vars=['2020', '2021', '2022', '2023', '2024', '2025'],
    var_name='year',
    value_name='tert_edu'
)

education_long['year'] = education_long['year'].astype(int)

education_long['tert_edu'] = pd.to_numeric(
    education_long['tert_edu'], 
    errors='coerce'
)

education_long = education_long[
    education_long['year'].isin([2021, 2022, 2023, 2024, 2025])
].copy()

print(education_long.head())

    nace_r2 geo  year  tert_edu
569       A  AT  2021      22.2
570       A  BE  2021      28.7
571       A  BG  2021       7.5
572       A  CH  2021      23.4
573       A  CY  2021      21.5


In [63]:
education_long.to_csv('data_panel/educ.csv', index = False)

## 9. High Skills

Share of employment in high-skill occupations — ISCO-08 major groups 1–3: managers, professionals and technicians (`lfsa_eisn2`).


In [65]:
occup = eurostat.get_data_df('lfsa_eisn2')
# print(occup)

In [66]:
occup.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
occup.drop(['2008', '2009', '2010', '2011','2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
occup.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54934 entries, 0 to 54933
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     54934 non-null  object 
 1   age      54934 non-null  object 
 2   sex      54934 non-null  object 
 3   nace_r2  54934 non-null  object 
 4   isco08   54934 non-null  object 
 5   unit     54934 non-null  object 
 6   geo      54934 non-null  object 
 7   2020     28538 non-null  float64
 8   2021     29340 non-null  float64
 9   2022     29531 non-null  float64
 10  2023     28976 non-null  float64
 11  2024     29068 non-null  float64
 12  2025     28940 non-null  float64
dtypes: float64(6), object(7)
memory usage: 5.4+ MB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\219748.py:1: SyntaxWarning: invalid escape sequence '\T'
  occup.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [67]:
# Filter 
# unit - Ths_per, thousandpersons
occup = occup.copy()
occup = occup[
    (occup['sex'] == 'T')& # for all genders
    (occup['age'] == 'Y20-64') # From 20 to 64 years
]

occup = occup.drop(columns=['freq', 'unit', 'age', 'sex'])
occup = occup[occup['geo'].isin(EU_EFTA)]
# print(occup)

In [68]:
occup['nace_r2_1d'] = occup['nace_r2'].map(nace_section_or_nan)
occup.drop(columns=['nace_r2'], inplace=True) 
occup.rename(columns={'nace_r2_1d' : 'nace_r2'}, inplace=True)

occup = occup.dropna()
# print(occup)

In [70]:
high_skill_codes = ['OC1', 'OC2', 'OC3']
total_code = 'TOTAL'
relevant_codes = high_skill_codes + [total_code]

occup_filtered = occup[occup['isco08'].isin(relevant_codes)].copy()

df_melted = occup_filtered.melt(
    id_vars=['geo', 'nace_r2', 'isco08'],          
    value_vars=['2020','2021', '2022', '2023', '2024', '2025'], 
    var_name='year_raw',
    value_name='emp_value'
)

df_melted['year'] = df_melted['year_raw'].str.extract(r'(\d+)').astype(int)

occup_pivoted = df_melted.pivot_table(
    index=['geo', 'nace_r2', 'year'], 
    columns='isco08',
    values='emp_value',
    aggfunc='sum'
).reset_index()


occup_pivoted.rename(columns={
    'OC1': 'OC1_Managers',
    'OC2': 'OC2_Professionals',
    'OC3': 'OC3_Technicians',
    'TOTAL': 'TOTAL_Occupied'
}, inplace=True)


occup_pivoted['total_high_skill'] = (
    occup_pivoted['OC1_Managers'] + 
    occup_pivoted['OC2_Professionals'] + 
    occup_pivoted['OC3_Technicians']
)



occup_pivoted['share_high_skill'] = (
    occup_pivoted['total_high_skill'] / occup_pivoted['TOTAL_Occupied']
) * 100



share_high_skill_df = occup_pivoted[['geo', 'nace_r2', 'year', 'share_high_skill']].copy()

share_high_skill_df = share_high_skill_df.dropna()

print("High Skill Share Calculation Head (Panel):")
print(share_high_skill_df.head())

High Skill Share Calculation Head (Panel):
isco08 geo nace_r2  year  share_high_skill
12      AT       C  2020         35.562831
13      AT       C  2021         36.579294
14      AT       C  2022         37.320574
15      AT       C  2023         38.711970
16      AT       C  2024         40.126825


In [71]:
share_high_skill_df.to_csv('data_panel/share_high_skill.csv', index = False)

## 10. Master Panel — Core Variables

Merges the sector-level variables into the core panel, without the macroeconomic controls.


In [ ]:
# education_long
# df_fsi_final
# prod_long
# wg_panel
# df_ict_panel
# df_train_panel
# df_ai_panel
# share_high_skill_df

### 10.1 Inspect the individual panels


In [72]:
print(df_ai_panel)

     geo nace_r2  ai_adoption  year
0     AT       C        9.610  2021
1     AT       C       10.960  2022
2     AT       C       12.310  2023
3     AT       C       22.710  2024
4     AT       C       32.540  2025
5     AT       F        3.120  2021
6     AT       F        3.700  2022
7     AT       F        4.280  2023
8     AT       F        7.350  2024
9     AT       F       14.890  2025
10    AT       G        6.940  2021
11    AT       G        7.645  2022
12    AT       G        8.350  2023
13    AT       G       15.670  2024
14    AT       G       26.460  2025
15    AT       H        6.980  2021
16    AT       H        7.645  2022
17    AT       H        8.310  2023
18    AT       H       13.480  2024
19    AT       H       19.370  2025
20    AT       I        3.310  2021
21    AT       I        3.470  2022
22    AT       I        3.630  2023
23    AT       I       15.870  2024
24    AT       I       23.720  2025
25    AT       J       30.270  2021
26    AT       J       33.66

In [73]:
df_train_panel['training_ict'] = round(df_train_panel['training_ict'],2)
print(df_train_panel)

    geo nace_r2  training_ict  year
0    AT       C         20.37  2020
1    AT       C         25.60  2022
2    AT       C         26.06  2024
3    AT       D         41.66  2020
4    AT       D         50.06  2022
5    AT       D         48.11  2024
6    AT       E         23.76  2020
7    AT       E         28.55  2022
8    AT       E         26.70  2024
9    AT       F          9.65  2020
10   AT       F          6.91  2022
11   AT       F          9.11  2024
12   AT       G         17.43  2020
13   AT       G         22.01  2022
14   AT       G         19.51  2024
15   AT       H         10.97  2020
16   AT       H         14.76  2022
17   AT       H         12.55  2024
18   AT       I          5.46  2020
19   AT       I          5.11  2022
20   AT       I          5.97  2024
21   AT       J         76.00  2020
22   AT       J         69.36  2022
23   AT       J         65.84  2024
24   AT       L         23.03  2020
25   AT       L         27.67  2022
26   AT       L         21.7

In [74]:
df_ict_panel['spec_ict'] = round(df_ict_panel['spec_ict'],2)
print(df_ict_panel)

    geo nace_r2  spec_ict  year
0    AT       C     28.74  2020
1    AT       C     28.37  2022
2    AT       C     30.02  2024
3    AT       D     40.63  2020
4    AT       D     48.06  2022
5    AT       D     47.21  2024
6    AT       E     23.40  2020
7    AT       E     27.69  2022
8    AT       E     25.20  2024
9    AT       F      8.24  2020
10   AT       F      9.03  2022
11   AT       F      7.12  2024
12   AT       G     16.25  2020
13   AT       G     22.79  2022
14   AT       G     18.57  2024
15   AT       H     12.28  2020
16   AT       H     16.96  2022
17   AT       H     12.04  2024
18   AT       I      3.51  2020
19   AT       I      9.37  2022
20   AT       I      5.87  2024
21   AT       J     88.96  2020
22   AT       J     80.81  2022
23   AT       J     80.74  2024
24   AT       L     26.38  2020
25   AT       L     31.21  2022
26   AT       L     29.09  2024
27   AT       M     30.41  2020
28   AT       M     35.97  2022
29   AT       M     27.08  2024
30   AT 

In [75]:
print(education_long)

     nace_r2 geo  year  tert_edu
569        A  AT  2021      22.2
570        A  BE  2021      28.7
571        A  BG  2021       7.5
572        A  CH  2021      23.4
573        A  CY  2021      21.5
574        A  CZ  2021      11.3
575        A  DE  2021      23.1
576        A  DK  2021      15.2
577        A  EE  2021      21.4
578        A  EL  2021       7.6
579        A  ES  2021      11.5
580        A  FI  2021      19.6
581        A  FR  2021      25.1
582        A  HR  2021       8.9
583        A  HU  2021      11.6
584        A  IE  2021      27.4
585        A  IS  2021      19.5
586        A  IT  2021       4.7
587        A  LT  2021      18.1
588        A  LV  2021      16.1
589        A  NL  2021      22.0
590        A  NO  2021      20.0
591        A  PL  2021      15.0
592        A  RO  2021      10.9
593        A  SE  2021      30.1
594        A  SI  2021      29.0
595        A  SK  2021      17.1
596        B  BG  2021      21.3
597        B  ES  2021      31.7
598       

In [76]:
prod_long['log_prod'] = np.round(np.log(prod_long['productivity']), 2)
print(prod_long)

       geo nace_r2  year  productivity  log_prod
0       AT       A  2021         31.78      3.46
1       BE       A  2021         54.96      4.01
2       BG       A  2021          5.80      1.76
3       CH       A  2021         35.99      3.58
4       CY       A  2021         24.42      3.20
5       CZ       A  2021         29.43      3.38
6       DE       A  2021         47.95      3.87
7       DK       A  2021         56.76      4.04
8       EE       A  2021         34.38      3.54
9       EL       A  2021         12.92      2.56
10      ES       A  2021         42.53      3.75
11      FI       A  2021         61.24      4.11
12      FR       A  2021         51.64      3.94
13      HR       A  2021         18.29      2.91
14      HU       A  2021         30.52      3.42
15      IE       A  2021         42.27      3.74
16      IS       A  2021        134.83      4.90
17      IT       A  2021         36.02      3.58
18      LI       A  2021         40.00      3.69
19      LT       A  

c:\ProgramData\anaconda3\Lib\site-packages\pandas\core\arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [77]:
wg_panel['real_wage'] = round(wg_panel['real_wage'],2)
wg_panel['log_wage'] = np.round(np.log(wg_panel['real_wage']), 2)
print(wg_panel)

     geo nace_r2  real_wage  year  log_wage
0     AT       B      26.18  2020      3.26
1     AT       B      25.84  2021      3.25
2     AT       B      24.12  2022      3.18
3     AT       B      24.31  2023      3.19
4     AT       B      25.86  2024      3.25
5     AT       B      26.19  2025      3.27
6     AT       C      27.10  2020      3.30
7     AT       C      26.74  2021      3.29
8     AT       C      26.18  2022      3.26
9     AT       C      26.07  2023      3.26
10    AT       C      26.90  2024      3.29
11    AT       C      27.12  2025      3.30
12    AT       D      38.81  2020      3.66
13    AT       D      37.23  2021      3.62
14    AT       D      36.43  2022      3.60
15    AT       D      36.04  2023      3.58
16    AT       D      37.18  2024      3.62
17    AT       D      37.34  2025      3.62
18    AT       E      22.49  2020      3.11
19    AT       E      21.44  2021      3.07
20    AT       E      20.98  2022      3.04
21    AT       E      20.94  202

In [78]:
df_fsi_final['FSI'] = round(df_fsi_final['FSI'],2)
print(df_fsi_final)

size_emp geo nace_r2  year    FSI
0         AT       B  2021   0.00
1         AT       B  2022   0.00
2         AT       B  2023   0.00
3         AT       B  2024  39.21
4         AT       C  2021  55.82
5         AT       C  2022  56.10
6         AT       C  2023  57.52
7         AT       C  2024  57.66
8         AT       D  2021   0.00
9         AT       D  2022   0.00
10        AT       D  2023   0.00
11        AT       D  2024  65.60
12        AT       E  2021  37.49
13        AT       E  2022  37.63
14        AT       E  2023  37.85
15        AT       E  2024  36.86
16        AT       F  2021  21.48
17        AT       F  2022  21.45
18        AT       F  2023  22.05
19        AT       F  2024  22.95
20        AT       G  2021  37.84
21        AT       G  2022  38.04
22        AT       G  2023  38.56
23        AT       G  2024  38.76
24        AT       H  2021  45.80
25        AT       H  2022  44.78
26        AT       H  2023  45.65
27        AT       H  2024  46.94
28        AT  

In [79]:
share_high_skill_df['share_high_skill'] = round(share_high_skill_df['share_high_skill'],2)
print(share_high_skill_df)

isco08 geo nace_r2  year  share_high_skill
12      AT       C  2020             35.56
13      AT       C  2021             36.58
14      AT       C  2022             37.32
15      AT       C  2023             38.71
16      AT       C  2024             40.13
17      AT       C  2025             42.41
30      AT       F  2020             26.18
31      AT       F  2021             25.30
32      AT       F  2022             27.47
33      AT       F  2023             28.69
34      AT       F  2024             29.30
35      AT       F  2025             28.70
36      AT       G  2020             25.14
37      AT       G  2021             25.07
38      AT       G  2022             25.97
39      AT       G  2023             28.12
40      AT       G  2024             26.42
41      AT       G  2025             27.20
42      AT       H  2020             24.49
43      AT       H  2021             23.55
44      AT       H  2022             24.20
45      AT       H  2023             22.09
46      AT 

### 10.2 Merge into the core master panel


In [ ]:
# BUILD THE NORMAL DATASET (Dependent & Control Variables for t)
# This includes Wages, AI Adoption, and Education (Years: 2021, 2023, 2025)

df_normal = wg_panel.copy()
df_normal = pd.merge(df_normal, df_ai_panel, on=['geo', 'nace_r2', 'year'], how='left')
# df_normal = pd.merge(df_normal, share_high_skill_df, on=['geo', 'nace_r2', 'year'], how='left')
df_normal = pd.merge(df_normal, education_long, on=['geo', 'nace_r2', 'year'], how='left')
df_normal = pd.merge(df_normal, prod_long, on=['geo', 'nace_r2', 'year'], how='left')
df_normal = pd.merge(df_normal, df_fsi_final, on=['geo', 'nace_r2', 'year'], how='left')

# BUILD THE LAGGED DATASET (Independent Variables for t-1)
# This includes ICT Specialists and Training (Years: 2020, 2022, 2024)

df_lag = df_ict_panel.copy()
df_lag = pd.merge(df_lag, df_train_panel, on=['geo', 'nace_r2', 'year'], how='left')

# ALIGN THE TIMELINES
# We add +1 to the lagged year so it matches the normal year during the merge.
# (e.g., 2020 becomes 2021, so it joins with the 2021 wage data)
df_lag['year'] = df_lag['year'] + 1

# Merge the lagged data into the normal dataset
df_master = pd.merge(df_normal, df_lag, on=['geo', 'nace_r2', 'year'], how='left')

# Drop any rows where 'year' is not in the target periods (if any stray years snuck in)
target_years = [2021, 2023, 2025]
df_master = df_master[df_master['year'].isin(target_years)].reset_index(drop=True)

print("\n--- MASTER PANEL DATASET (Lagged Structure) ---")
print(df_master['year'].value_counts().sort_index()) 
print(df_master.head())


--- MASTER PANEL DATASET (Lagged Structure) ---
year
2021    473
2023    473
2025    473
Name: count, dtype: int64
  geo nace_r2  real_wage  year  log_wage  ai_adoption  tert_edu  productivity  \
0  AT       B      25.84  2021      3.25          NaN       NaN        169.97   
1  AT       B      24.31  2023      3.19          NaN       NaN        175.68   
2  AT       B      26.19  2025      3.27          NaN       NaN           NaN   
3  AT       C      26.74  2021      3.29         9.61      31.8        102.25   
4  AT       C      26.07  2023      3.26        12.31      33.5        108.93   

   log_prod    FSI  spec_ict  training_ict  
0      5.14   0.00       NaN           NaN  
1      5.17   0.00       NaN           NaN  
2       NaN    NaN       NaN           NaN  
3      4.63  55.82     28.74         20.37  
4      4.69  57.52     28.37         25.60  


In [ ]:
# Create a strict dataset dropping any row that has an NA in any column
df_regression = df_master.dropna(subset=['ai_adoption', 'tert_edu'])
final_n = len(df_regression)
print(f"Total starting rows: {len(df_master)}")
print(f"Final usable rows for regression: {final_n}")

print("\nUsable rows by period:")
print(df_regression['year'].value_counts().sort_index())

Total starting rows: 1419
Final usable rows for regression: 708

Usable rows by period:
year
2021    236
2023    236
2025    236
Name: count, dtype: int64


In [124]:
missing_by_year = (
    df_regression.groupby('year')
    .apply(lambda g: g.isna().sum())
    .drop(columns='year', errors='ignore')
)
print("--- Missing entries (count) per variable, by year ---")
print(missing_by_year.T)

--- Missing entries (count) per variable, by year ---
year          2021  2023  2025
geo              0     0     0
nace_r2          0     0     0
real_wage        0     0     0
log_wage         0     0     0
ai_adoption      0     0     0
tert_edu         0     0     0
productivity     0     0   195
log_prod         0     0   195
FSI              0     0   236
spec_ict         0     0     0
training_ict     0     0     0


C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\469068950.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.isna().sum())


In [125]:

df_regression.to_csv('data_panel/panel_master.csv', index = False )

## 11. Macroeconomic Controls


### 11.1 Unemployment rate


In [126]:
unempl = eurostat.get_data_df('tps00203')
print(unempl)

    freq     age     unit sex geo\TIME_PERIOD     2014     2015     2016  \
0      A  Y15-74   PC_ACT   T              AT      6.0      6.1      6.5   
1      A  Y15-74   PC_ACT   T              BA      NaN      NaN      NaN   
2      A  Y15-74   PC_ACT   T              BE      8.7      8.7      7.9   
3      A  Y15-74   PC_ACT   T              BG     12.4     10.1      8.6   
4      A  Y15-74   PC_ACT   T              CH      4.9      4.8      5.0   
5      A  Y15-74   PC_ACT   T              CY     16.1     15.0     13.0   
6      A  Y15-74   PC_ACT   T              CZ      6.1      5.1      4.0   
7      A  Y15-74   PC_ACT   T              DE      4.7      4.4      3.9   
8      A  Y15-74   PC_ACT   T              DK      6.9      6.3      6.0   
9      A  Y15-74   PC_ACT   T            EA20     11.7     11.0     10.1   
10     A  Y15-74   PC_ACT   T            EA21     11.8     11.0     10.1   
11     A  Y15-74   PC_ACT   T              EE      7.3      6.4      6.8   
12     A  Y1

In [127]:
unempl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
unempl.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 114 entries, 0 to 113
Data columns (total 17 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   freq    114 non-null    object 
 1   age     114 non-null    object 
 2   unit    114 non-null    object 
 3   sex     114 non-null    object 
 4   geo     114 non-null    object 
 5   2014    111 non-null    float64
 6   2015    111 non-null    float64
 7   2016    111 non-null    float64
 8   2017    111 non-null    float64
 9   2018    111 non-null    float64
 10  2019    111 non-null    float64
 11  2020    111 non-null    float64
 12  2021    111 non-null    float64
 13  2022    111 non-null    float64
 14  2023    111 non-null    float64
 15  2024    111 non-null    float64
 16  2025    111 non-null    float64
dtypes: float64(12), object(5)
memory usage: 15.3+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\1391666431.py:1: SyntaxWarning: invalid escape sequence '\T'
  unempl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [128]:
unempl.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
print(unempl)

    freq     age     unit sex        geo     2020     2021     2022     2023  \
0      A  Y15-74   PC_ACT   T         AT      6.0      6.2      4.8      5.1   
1      A  Y15-74   PC_ACT   T         BA      NaN     17.4     15.4     13.2   
2      A  Y15-74   PC_ACT   T         BE      5.8      6.3      5.6      5.5   
3      A  Y15-74   PC_ACT   T         BG      6.1      5.2      4.2      4.3   
4      A  Y15-74   PC_ACT   T         CH      4.8      5.1      4.1      4.1   
5      A  Y15-74   PC_ACT   T         CY      7.6      7.2      6.3      5.8   
6      A  Y15-74   PC_ACT   T         CZ      2.6      2.8      2.2      2.6   
7      A  Y15-74   PC_ACT   T         DE      3.6      3.6      3.1      3.1   
8      A  Y15-74   PC_ACT   T         DK      5.6      5.1      4.5      5.1   
9      A  Y15-74   PC_ACT   T       EA20      8.0      7.8      6.8      6.6   
10     A  Y15-74   PC_ACT   T       EA21      7.9      7.7      6.7      6.5   
11     A  Y15-74   PC_ACT   T         EE

In [ ]:
# Filter 
# Y15-74 default age class
# sex total is default

unempl_r = unempl.copy()
unempl_r = unempl_r[ 
    (unempl_r['unit'] == 'PC_ACT') # Percentage of population in the labour force
]

unempl_r = unempl_r.drop(columns=['freq', 'unit', 'age', 'sex'])
unempl_r = unempl_r[unempl_r['geo'].isin(EU_EFTA)]
print(unempl_r)

   geo  2020  2021  2022  2023  2024  2025
0   AT   6.0   6.2   4.8   5.1   5.2   5.7
2   BE   5.8   6.3   5.6   5.5   5.7   6.2
3   BG   6.1   5.2   4.2   4.3   4.2   3.5
4   CH   4.8   5.1   4.1   4.1   4.4   4.9
5   CY   7.6   7.2   6.3   5.8   4.8   4.4
6   CZ   2.6   2.8   2.2   2.6   2.6   2.8
7   DE   3.6   3.6   3.1   3.1   3.5   3.8
8   DK   5.6   5.1   4.5   5.1   6.2   6.4
11  EE   6.9   6.2   5.6   6.4   7.6   7.5
12  EL  17.6  14.7  12.5  11.1  10.1   8.9
13  ES  15.5  14.9  13.0  12.2  11.4  10.5
15  FI   7.7   7.7   6.8   7.2   8.4   9.7
16  FR   8.0   7.9   7.3   7.3   7.4   7.7
17  HR   7.4   7.5   6.8   6.1   5.0   4.9
18  HU   4.1   4.0   3.6   4.1   4.5   4.4
19  IE   5.9   6.2   4.5   4.3   4.3   4.7
20  IS   5.5   6.1   3.8   3.5   3.6   4.5
21  IT   9.3   9.5   8.1   7.7   6.5   6.1
22  LT   8.5   7.1   6.0   6.9   7.1   6.9
23  LU   6.8   5.3   4.6   5.2   6.4   6.5
24  LV   8.1   7.6   6.9   6.5   6.9   6.9
27  MT   4.9   3.8   3.5   3.5   3.2   3.1
28  NL   4.

In [130]:
unempl_r = unempl_r.dropna()
print(unempl_r)

   geo  2020  2021  2022  2023  2024  2025
0   AT   6.0   6.2   4.8   5.1   5.2   5.7
2   BE   5.8   6.3   5.6   5.5   5.7   6.2
3   BG   6.1   5.2   4.2   4.3   4.2   3.5
4   CH   4.8   5.1   4.1   4.1   4.4   4.9
5   CY   7.6   7.2   6.3   5.8   4.8   4.4
6   CZ   2.6   2.8   2.2   2.6   2.6   2.8
7   DE   3.6   3.6   3.1   3.1   3.5   3.8
8   DK   5.6   5.1   4.5   5.1   6.2   6.4
11  EE   6.9   6.2   5.6   6.4   7.6   7.5
12  EL  17.6  14.7  12.5  11.1  10.1   8.9
13  ES  15.5  14.9  13.0  12.2  11.4  10.5
15  FI   7.7   7.7   6.8   7.2   8.4   9.7
16  FR   8.0   7.9   7.3   7.3   7.4   7.7
17  HR   7.4   7.5   6.8   6.1   5.0   4.9
18  HU   4.1   4.0   3.6   4.1   4.5   4.4
19  IE   5.9   6.2   4.5   4.3   4.3   4.7
20  IS   5.5   6.1   3.8   3.5   3.6   4.5
21  IT   9.3   9.5   8.1   7.7   6.5   6.1
22  LT   8.5   7.1   6.0   6.9   7.1   6.9
23  LU   6.8   5.3   4.6   5.2   6.4   6.5
24  LV   8.1   7.6   6.9   6.5   6.9   6.9
27  MT   4.9   3.8   3.5   3.5   3.2   3.1
28  NL   4.

In [131]:
# cols_to_melt = ['2020', '2021', '2022', '2023', '2024', '2025']
cols_to_melt = ['2021', '2023', '2025']

unempl_r_panel = unempl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='unempl_r'           
)

unempl_r_panel['year'] = unempl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
unempl_r_panel.drop(columns=['year_raw'], inplace=True)
unempl_r_panel = unempl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(unempl_r_panel.head(10))

  geo  unempl_r  year
0  AT       6.2  2021
1  AT       5.1  2023
2  AT       5.7  2025
3  BE       6.3  2021
4  BE       5.5  2023
5  BE       6.2  2025
6  BG       5.2  2021
7  BG       4.3  2023
8  BG       3.5  2025
9  CH       5.1  2021


In [132]:
unempl_r_panel.to_csv('data_panel/unempl_r.csv', index = False)

### 11.2 Inflation rate


In [133]:
infl = eurostat.get_data_df('tec00118')
infl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
infl.drop(['2014', '2015', '2016', '2017', '2018', '2019'], axis='columns', inplace=True)
infl.info()

<>:2: SyntaxWarning: invalid escape sequence '\T'
<>:2: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\3800707276.py:2: SyntaxWarning: invalid escape sequence '\T'
  infl.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 10 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   freq      42 non-null     object 
 1   unit      42 non-null     object 
 2   coicop18  42 non-null     object 
 3   geo       42 non-null     object 
 4   2020      41 non-null     float64
 5   2021      41 non-null     float64
 6   2022      41 non-null     float64
 7   2023      41 non-null     float64
 8   2024      41 non-null     float64
 9   2025      40 non-null     float64
dtypes: float64(6), object(4)
memory usage: 3.4+ KB


In [134]:
# Filter 

infl_r = infl.drop(columns=['freq', 'unit', 'coicop18'])
infl_r = infl_r[infl_r['geo'].isin(EU_EFTA)]
print(infl_r)

   geo  2020  2021  2022  2023  2024  2025
1   AT   1.4   2.8   8.6   7.7   2.9   3.6
2   BE   0.4   3.2  10.3   2.3   4.3   3.0
3   BG   1.2   2.8  13.0   8.6   2.6   3.5
4   CH  -0.8   0.5   2.7   2.3   1.1   0.1
5   CY  -1.1   2.3   8.1   3.9   2.3   0.8
6   CZ   3.3   3.3  14.8  12.0   2.7   2.3
7   DE   0.4   3.2   8.7   6.0   2.5   2.3
8   DK   0.3   1.9   8.6   3.4   1.3   1.8
12  EE  -0.6   4.5  19.4   9.1   3.7   4.8
13  EL  -1.3   0.6   9.3   4.2   3.0   2.9
14  ES  -0.3   3.0   8.3   3.4   2.9   2.7
16  FI   0.4   2.1   7.2   4.3   1.0   1.8
17  FR   0.5   2.1   5.9   5.7   2.3   0.9
18  HR   0.0   2.7  10.7   8.4   4.0   4.4
19  HU   3.4   5.2  15.3  17.0   3.7   4.4
20  IE  -0.5   2.4   8.1   5.2   1.3   2.1
21  IS   1.2   3.7   5.7   8.0   4.5   3.7
22  IT  -0.2   2.0   8.7   5.9   1.1   1.6
23  LT   1.1   4.6  18.9   8.7   0.9   3.4
24  LU   0.0   3.5   8.2   2.9   2.3   2.5
25  LV   0.1   3.2  17.2   9.1   1.3   3.8
28  MT   0.8   0.7   6.1   5.6   2.4   2.4
29  NL   1.

In [135]:
infl_r = infl_r.dropna()
print(infl_r)

   geo  2020  2021  2022  2023  2024  2025
1   AT   1.4   2.8   8.6   7.7   2.9   3.6
2   BE   0.4   3.2  10.3   2.3   4.3   3.0
3   BG   1.2   2.8  13.0   8.6   2.6   3.5
4   CH  -0.8   0.5   2.7   2.3   1.1   0.1
5   CY  -1.1   2.3   8.1   3.9   2.3   0.8
6   CZ   3.3   3.3  14.8  12.0   2.7   2.3
7   DE   0.4   3.2   8.7   6.0   2.5   2.3
8   DK   0.3   1.9   8.6   3.4   1.3   1.8
12  EE  -0.6   4.5  19.4   9.1   3.7   4.8
13  EL  -1.3   0.6   9.3   4.2   3.0   2.9
14  ES  -0.3   3.0   8.3   3.4   2.9   2.7
16  FI   0.4   2.1   7.2   4.3   1.0   1.8
17  FR   0.5   2.1   5.9   5.7   2.3   0.9
18  HR   0.0   2.7  10.7   8.4   4.0   4.4
19  HU   3.4   5.2  15.3  17.0   3.7   4.4
20  IE  -0.5   2.4   8.1   5.2   1.3   2.1
21  IS   1.2   3.7   5.7   8.0   4.5   3.7
22  IT  -0.2   2.0   8.7   5.9   1.1   1.6
23  LT   1.1   4.6  18.9   8.7   0.9   3.4
24  LU   0.0   3.5   8.2   2.9   2.3   2.5
25  LV   0.1   3.2  17.2   9.1   1.3   3.8
28  MT   0.8   0.7   6.1   5.6   2.4   2.4
29  NL   1.

In [136]:
infl_r_panel = infl_r.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='infl_r'           
)

infl_r_panel['year'] = infl_r_panel['year_raw'].str.extract(r'(\d+)').astype(int)
infl_r_panel.drop(columns=['year_raw'], inplace=True)
infl_r_panel = infl_r_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(infl_r_panel.head(10))

  geo  infl_r  year
0  AT     2.8  2021
1  AT     7.7  2023
2  AT     3.6  2025
3  BE     3.2  2021
4  BE     2.3  2023
5  BE     3.0  2025
6  BG     2.8  2021
7  BG     8.6  2023
8  BG     3.5  2025
9  CH     0.5  2021


In [137]:
infl_r_panel.to_csv('data_panel/infl_r.csv', index = False)

### 11.3 GDP per capita


In [ ]:
gdp_pc = eurostat.get_data_df('nama_10_pc')
print(gdp_pc)

In [139]:
gdp_pc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)
columns_to_drop = gdp_pc.columns[4:49]
gdp_pc.drop(columns=columns_to_drop, inplace=True)
gdp_pc.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4509 entries, 0 to 4508
Data columns (total 10 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   freq     4509 non-null   object 
 1   unit     4509 non-null   object 
 2   na_item  4509 non-null   object 
 3   geo      4509 non-null   object 
 4   2020     4509 non-null   float64
 5   2021     4509 non-null   float64
 6   2022     4500 non-null   float64
 7   2023     4392 non-null   float64
 8   2024     4389 non-null   float64
 9   2025     4105 non-null   float64
dtypes: float64(6), object(4)
memory usage: 352.4+ KB


<>:1: SyntaxWarning: invalid escape sequence '\T'
<>:1: SyntaxWarning: invalid escape sequence '\T'
C:\Users\ydmar\AppData\Local\Temp\ipykernel_14260\871698340.py:1: SyntaxWarning: invalid escape sequence '\T'
  gdp_pc.rename(columns={"geo\TIME_PERIOD": "geo"}, inplace=True)


In [ ]:
gdp_pc = gdp_pc.dropna()
print(gdp_pc)

In [ ]:
gdp = gdp_pc.copy()
gdp = gdp[
    (gdp['na_item'] == 'B1GQ')& # Gross domestic product at market prices
    (gdp['unit'] == 'CP_EUR_HAB') # Current price, euro per capita
]


gdp = gdp.drop(columns=['freq', 'unit', 'na_item'])
gdp = gdp[gdp['geo'].isin(EU_EFTA)]
print(gdp)

     geo      2020      2021      2022      2023      2024      2025
2626  AT   42650.0   45380.0   49640.0   52330.0   53830.0   55870.0
2628  BE   40190.0   43680.0   48090.0   51090.0   52310.0   53930.0
2629  BG    9440.0   10960.0   13310.0   14660.0   16260.0   18060.0
2630  CH   76710.0   81610.0   92900.0   96280.0   99430.0  101810.0
2631  CY   24630.0   27850.0   31560.0   33870.0   35670.0   36850.0
2632  CZ   20980.0   23430.0   26670.0   29330.0   29510.0   31880.0
2633  DE   42020.0   44910.0   48340.0   50660.0   51830.0   53520.0
2634  DK   53540.0   58640.0   64430.0   62760.0   66970.0   69550.0
2640  EE   20960.0   23650.0   27260.0   28080.0   28990.0   30380.0
2641  EL   15660.0   17350.0   19570.0   21300.0   22480.0   23570.0
2642  ES   23850.0   26090.0   28790.0   30980.0   32630.0   34210.0
2644  FI   42740.0   44890.0   47890.0   48950.0   49130.0   49850.0
2645  FR   34230.0   36910.0   38870.0   41340.0   42680.0   43350.0
2646  HR   12980.0   15070.0   175

In [142]:
gdp_panel = gdp.melt(
    id_vars=['geo'],     
    value_vars=cols_to_melt,       
    var_name='year_raw',            
    value_name='gdp'           
)

gdp_panel['year'] = gdp_panel['year_raw'].str.extract(r'(\d+)').astype(int)
gdp_panel.drop(columns=['year_raw'], inplace=True)
gdp_panel = gdp_panel.sort_values(by=['geo', 'year']).reset_index(drop=True)
print(gdp_panel.head(10))

  geo      gdp  year
0  AT  45380.0  2021
1  AT  52330.0  2023
2  AT  55870.0  2025
3  BE  43680.0  2021
4  BE  51090.0  2023
5  BE  53930.0  2025
6  BG  10960.0  2021
7  BG  14660.0  2023
8  BG  18060.0  2025
9  CH  81610.0  2021


In [143]:
gdp_panel.to_csv('data_panel/gdp.csv', index = False)

## 12. Full Dataset

Combines the core panel with the macroeconomic controls and adds the log of GDP per capita.


In [144]:
# unempl_r_panel
# infl_r_panel
# gdp_panel

df_full_master = df_regression.copy()
df_full_master = pd.merge(df_full_master, unempl_r_panel, on=['geo', 'year'], how='left')
df_full_master = pd.merge(df_full_master, infl_r_panel, on=['geo', 'year'], how='left')
df_full_master = pd.merge(df_full_master, gdp_panel, on=['geo', 'year'], how='left')
df_full_master['log_gdp'] = np.round(np.log(df_full_master['gdp']), 2)

print("\n--- MASTER PANEL FULL DATASET (2021-2025) ---")
print(df_full_master['year'].value_counts().sort_index()) # Verifies only 2021, 2023, 2025 exist
print(df_full_master)


--- MASTER PANEL FULL DATASET (2021-2025) ---
year
2021    236
2023    236
2025    236
Name: count, dtype: int64
    geo nace_r2  real_wage  year  log_wage  ai_adoption  tert_edu  \
0    AT       C      26.74  2021      3.29         9.61      31.8   
1    AT       C      26.07  2023      3.26        12.31      33.5   
2    AT       C      27.12  2025      3.30        32.54      36.2   
3    AT       F      22.43  2021      3.11         3.12      21.2   
4    AT       F      20.32  2023      3.01         4.28      22.5   
5    AT       F      20.43  2025      3.02        14.89      25.6   
6    AT       G      21.35  2021      3.06         6.94      23.1   
7    AT       G      20.48  2023      3.02         8.35      24.5   
8    AT       G      21.65  2025      3.08        26.46      22.4   
9    AT       H      21.71  2021      3.08         6.98      21.8   
10   AT       H      22.01  2023      3.09         8.31      23.4   
11   AT       H      22.66  2025      3.12        19.37   

In [145]:
print(df_full_master.isna().values.any())

True


In [146]:
df_full_master.to_csv('data_panel/full_panel_master.csv', index = False )